# FTEC 5660 Homework 2 part2
## Name: LI Shaoxuan

Let's create a social media account for your agent

# Setup your agent

In [ ]:

# 📦 Install Required Packages
!pip install langchain-google-genai langchain-core langchain-experimental
!pip install yfinance


In [3]:

#  API Key Setup
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')
assert GEMINI_VERTEX_API_KEY, "Please set your VERTEX_API_KEY in Colab secrets"

In [4]:

#  Initialize Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_VERTEX_API_KEY,
    vertexai=True,
    temperature=0
)

# Create a moltbook account for your agent

In [5]:
# This function is used to encode your student id to ensure the privacy

def encode_student_id(student_id: int) -> str:
    """
    Reversibly encode a student ID using an affine cipher.

    Args:
        student_id (int): Original student ID (non-negative integer)

    Returns:
        str: Encoded ID as a zero-padded string
    """
    if student_id < 0:
        raise ValueError("student_id must be non-negative")

    M = 10**8
    a = 137
    b = 911

    encoded = (a * student_id + b) % M
    return f"{encoded:08d}"

In [6]:
# Before creating your agent please encode your student id using this function and replace XXXX by the encoded number
encode_student_id(1155241119)

'68034214'

In [7]:
# Please use the encoded student id
!curl -X POST https://www.moltbook.com/api/v1/agents/register \
  -H "Content-Type: application/json" \
  -d '{"name": "5xl_68034214", "description": "Va"}'

{"success":true,"message":"Welcome to Moltbook! 🦞","agent":{"id":"79bcf56f-3446-4ca7-954a-c6c3e22d56dc","name":"5xl_68034214","api_key":"moltbook_sk_7GU-_TOFDTYhCJ0CfhJOtF3BDUQiBKXs","claim_url":"https://www.moltbook.com/claim/moltbook_claim_62HseGJeqlVWrYyGPonlwW8nPiderNkt","verification_code":"shell-4JAL","profile_url":"https://www.moltbook.com/u/5xl_68034214","created_at":"2026-02-27T07:57:13.934Z"},"setup":{"step_1":{"action":"SAVE YOUR API KEY","details":"Store it securely - you need it for all requests and it cannot be retrieved later!","critical":true},"step_2":{"action":"SET UP HEARTBEAT","details":"Add HEARTBEAT.md to your heartbeat routine so you check Moltbook periodically","url":"https://www.moltbook.com/heartbeat.md","why":"Without this, you'll never know when you're claimed or when someone replies to you!"},"step_3":{"action":"TELL YOUR HUMAN","details":"Send them the claim URL so they can verify you","message_template":"Hey! I just signed up for Moltbook, the social netw

- After sucessfully register, you will see a notification of the format:

"success":true,"message":"Welcome to Moltbook! 🦞","agent":"id":"...","name":"...","api_key":"...", "claim_url": "..."

- Please save your the api key as MOLTBOOK_API_KEY in the Secrets section of your Colab.
- Then you complete the registration by accessing the claim_url and follow the guideline in the url.

# supplementary tools added
`read_api_docs` and `subscribe_submolt`

In [8]:
# Create a tool set to interact with moltbook

import os
import requests
from langchain_core.tools import tool

MOLTBOOK_API_KEY = userdata.get('MOLTBOOK_API_KEY')
BASE_URL = "https://www.moltbook.com/api/v1"

HEADERS = {
    "Authorization": f"Bearer {MOLTBOOK_API_KEY}",
    "Content-Type": "application/json"
}

# ---------- FEED ----------
@tool
def get_feed(sort: str = "new", limit: int = 10) -> dict:
    """Fetch Moltbook feed."""
    r = requests.get(
        f"{BASE_URL}/feed",
        headers=HEADERS,
        params={"sort": sort, "limit": limit},
        timeout=15
    )
    return r.json()

# ---------- SEARCH ----------
@tool
def search_moltbook(query: str, type: str = "all") -> dict:
    """Semantic search Moltbook posts, comments, agents."""
    r = requests.get(
        f"{BASE_URL}/search",
        headers=HEADERS,
        params={"q": query, "type": type},
        timeout=15
    )
    return r.json()

# ---------- POST ----------
@tool
def create_post(submolt: str, title: str, content: str) -> dict:
    """Create a new text post."""
    payload = {
        "submolt": submolt,
        "title": title,
        "content": content
    }
    r = requests.post(
        f"{BASE_URL}/posts",
        headers=HEADERS,
        json=payload,
        timeout=15
    )
    return r.json()

# ---------- COMMENT ----------
@tool
def comment_post(post_id: str, content: str) -> dict:
    """Comment on a post."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/comments",
        headers=HEADERS,
        json={"content": content},
        timeout=15
    )
    return r.json()

# ---------- VOTE ----------
@tool
def upvote_post(post_id: str) -> dict:
    """Upvote a post."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/upvote",
        headers=HEADERS,
        timeout=15
    )
    return r.json()


# supplementary tools added
# ---------- READ DOCS ----------
@tool
def read_api_docs() -> str:
    """
    Fetch and read the Moltbook API documentation (skill.md).
    Call this FIRST to understand the required API endpoints before performing actions.
    """
    try:
        r = requests.get("https://www.moltbook.com/skill.md", timeout=15)
        return r.text
    except Exception as e:
        return f"Failed to fetch docs: {e}"

# ---------- SUBSCRIBE ----------
@tool
def subscribe_submolt(submolt_name: str) -> dict:
    """
    Subscribe to a specific submolt.
    Ensure you format the submolt_name correctly based on the API docs.
    """
    # Assuming a standard API path, if the Agent detects a difference in skill.md, it will know the cause of the error
    clean_name = submolt_name.replace("/m/", "").replace("m/", "")
    r = requests.post(
        f"{BASE_URL}/submolts/{clean_name}/subscribe",
        headers=HEADERS,
        timeout=15
    )
    return r.json()


# rewrite the prompt

In [9]:
# rewrite the prompt
SYSTEM_PROMPT = """
You are an autonomous Moltbook AI agent.

Your immediate mission is to complete the following tasks sequentially:
1. Use `read_api_docs` to read the official API documentation.
2. Subscribe to the submolt: /m/ftec5660
3. Upvote the specific post with ID: 47ff50f3-8255-4dee-87f4-2c3637c7351c
4. Comment on the specific post with ID: 47ff50f3-8255-4dee-87f4-2c3637c7351c with a short, highly professional, and insightful thought about AI/ML.

Rules:
1. ALWAYS read the API docs first to ensure you are using the correct endpoint logic.
2. Only comment if you add new insight.
3. If an endpoint fails, read the API docs again or adjust your parameters.
4. Prefer short, clear, professional language.

Available tools:
- get_feed
- search_moltbook
- create_post
- comment_post
- upvote_post
- read_api_docs
- subscribe_submolt
"""


# A simple agent to interact with moltbook

## add tools:
`read_api_docs`Endows Agents with the ability to read documents;  
`subscribe_submolt`Grants the Agent the ability to subscribe

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import ToolMessage
import time
import json
from datetime import datetime
from typing import Any

def log(section: str, message: str):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    print(f"[{ts}] [{section}] {message}")

def pretty(obj: Any, max_len: int = 800):
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    return text if len(text) <= max_len else text[:max_len] + "\n...<truncated>"

def moltbook_agent_loop(
    instruction: str | None = None,
    max_turns: int = 8,
    verbose: bool = True,
):
    log("INIT", "Starting Moltbook agent loop")

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        api_key=GEMINI_VERTEX_API_KEY,
        vertexai=True,
    )

    tools = [
        get_feed,
        search_moltbook,
        create_post,
        comment_post,
        upvote_post,
        read_api_docs,     # <- New addition: Endow Agents with the ability to read documents
        subscribe_submolt  # <- New addition: Grant the Agent the ability to subscribe
    ]

    agent = llm.bind_tools(tools)

    history = [("system", SYSTEM_PROMPT)]

    if instruction:
        history.append(("human", f"Human instruction: {instruction}"))
        log("HUMAN", instruction)
    else:
        history.append(("human", "Perform your Moltbook heartbeat check."))
        log("HEARTBEAT", "No human instruction – autonomous mode")

    # ================================
    # Main agent loop
    # ================================
    for turn in range(1, max_turns + 1):
        log("TURN", f"Turn {turn}/{max_turns} started")
        turn_start = time.time()

        response = agent.invoke(history)
        history.append(response)

        if verbose:
            log("LLM", "Model responded")
            log("LLM.CONTENT", response.content or "<empty>")
            log("LLM.TOOL_CALLS", pretty(response.tool_calls or []))

        # ============================
        # STOP CONDITION
        # ============================
        if not response.tool_calls:
            elapsed = round(time.time() - turn_start, 2)
            log("STOP", f"No tool calls — final answer produced in {elapsed}s")
            return response.content

        # ============================
        # TOOL EXECUTION
        # ============================
        for i, call in enumerate(response.tool_calls, start=1):
            tool_name = call["name"]
            args = call["args"]
            tool_id = call["id"]

            log("TOOL", f"[{i}] Calling `{tool_name}`")
            log("TOOL.ARGS", pretty(args))

            tool_fn = globals().get(tool_name)
            tool_start = time.time()

            try:
                result = tool_fn.invoke(args)
                status = "success"
            except Exception as e:
                result = {"error": str(e)}
                status = "error"

            tool_elapsed = round(time.time() - tool_start, 2)

            log(
                "TOOL.RESULT",
                f"{tool_name} finished ({status}) in {tool_elapsed}s"
            )

            if verbose:
                log("TOOL.OUTPUT", pretty(result))

            history.append(
                ToolMessage(
                    tool_call_id=tool_id,
                    content=str(result),
                )
            )

        turn_elapsed = round(time.time() - turn_start, 2)
        log("TURN", f"Turn {turn} completed in {turn_elapsed}s")

    # ================================
    # MAX TURNS REACHED
    # ================================
    log("STOP", "Max turns reached without final answer")
    return "Agent stopped after reaching max turns."



# FTEC5660 Hw2 Execution

In [11]:
# ================================
# FTEC5660 Homework 2 Execution
# ================================
# Instruct the agent to autonomously complete the homework requirements.
final_result = moltbook_agent_loop(
    "Execute your mission: read the docs, subscribe to /m/ftec5660, and upvote + comment on post 47ff50f3-8255-4dee-87f4-2c3637c7351c."
)

print("\n--- FINAL AGENT OUTPUT ---")
print(final_result)

/tmp/ipython-input-840/2032541230.py:9: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[08:30:02] [INIT] Starting Moltbook agent loop
[08:30:02] [HUMAN] Execute your mission: read the docs, subscribe to /m/ftec5660, and upvote + comment on post 47ff50f3-8255-4dee-87f4-2c3637c7351c.
[08:30:02] [TURN] Turn 1/8 started
[08:30:04] [LLM] Model responded
[08:30:04] [LLM.CONTENT] <empty>
[08:30:04] [LLM.TOOL_CALLS] [
  {
    "name": "read_api_docs",
    "args": {},
    "id": "c77add27-f297-419f-9ab8-82ae2b882d9a",
    "type": "tool_call"
  }
]
[08:30:04] [TOOL] [1] Calling `read_api_docs`
[08:30:04] [TOOL.ARGS] {}
[08:30:05] [TOOL.RESULT] read_api_docs finished (success) in 0.38s
[08:30:05] [TOOL.OUTPUT] "---\nname: moltbook\nversion: 1.12.0\ndescription: The social network for AI agents. Post, comment, upvote, and create communities.\nhomepage: https://www.moltbook.com\nmetadata: {\"moltbot\":{\"emoji\":\"🦞\",\"category\":\"social\",\"api_base\":\"https://www.moltbook.com/api/v1\"}}\n---\n\n# Moltbook\n\nThe social network for AI agents. Post, comment, upvote, and create commu

In [ ]:
# You need to complte the tool set so that your agent can find the submolt
# moltbook_agent_loop("find submolt named ftec5660")

